# EyeShield Single Model Evaluation

This notebook loads a single trained model checkpoint uploaded to Google Colab and evaluates it on a test dataset. It will calculate overall accuracy, macro F1-score, and display a confusion matrix.

## 1. Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import timm # Assuming timm was used for model creation, like efficientnet

## 2. Load Trained Model

Upload your model checkpoint (`.pth` file) to Colab, then specify the path below.

In [ ]:
# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "tf_efficientnet_b4_ns" # Adjust this to match your training configuration
NUM_CLASSES = 3 # Adjust this to match your number of classes (e.g., Normal, Mild, Severe)

# **UPDATE THIS PATH to point to your uploaded model checkpoint in Colab**
CHECKPOINT_PATH = "/content/best_fold_0.pth"

print(f"Using device: {DEVICE}")

# Initialize model architecture function
def create_model(model_name, num_classes):
    model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    return model

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading single model from {CHECKPOINT_PATH}...")
    model = create_model(MODEL_NAME, NUM_CLASSES)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
else:
    print(f"ERROR: Model checkpoint not found at {CHECKPOINT_PATH}!")
    print("Please upload your model file to the Colab folder and update the CHECKPOINT_PATH variable if needed.")

## 3. Load Evaluation Data

In [ ]:
# Define your evaluation dataset path and labels
TEST_CSV_PATH = "test_data.csv" # Update with your test csv path
TEST_IMAGE_DIR = "data/test_images/" # Update with your test image directory

# Define transformations (MUST match training validation transforms)
IMAGE_SIZE = 384 # Adjust to what was used during training
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class EyeShieldDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row['image_id'])
        # Handle file extensions appropriately based on your dataset
        if not img_id.endswith(('.jpg', '.png', '.jpeg')):
            img_id += '.jpg' 
            
        img_path = os.path.join(self.img_dir, img_id)
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        label = row['label'] # Ensure your CSV has a 'label' column
        return image, torch.tensor(label, dtype=torch.long)

# Create mock data for pipeline validation if CSV doesn't exist
if not os.path.exists(TEST_CSV_PATH):
    print(f"Warning: {TEST_CSV_PATH} not found. Using a dummy dataset for demonstration.")
    dummy_df = pd.DataFrame({'image_id': ['mock1.jpg', 'mock2.jpg'], 'label': [0, 1]})
    os.makedirs(TEST_IMAGE_DIR, exist_ok=True)
    Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color = 'red').save(os.path.join(TEST_IMAGE_DIR, 'mock1.jpg'))
    Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), color = 'blue').save(os.path.join(TEST_IMAGE_DIR, 'mock2.jpg'))
    test_dataset = EyeShieldDataset(dummy_df, TEST_IMAGE_DIR, transform=test_transform)
else:
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = EyeShieldDataset(test_df, TEST_IMAGE_DIR, transform=test_transform)

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)
print(f"Total test samples: {len(test_dataset)}")

## 4. Generate Predictions

In [ ]:
all_preds = []
all_targets = []
all_probs = []

print("Running inference on the test dataset...")
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(DEVICE)
        targets = targets.numpy()
        
        # Single model forward pass
        try:
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_targets.extend(targets)
        except NameError:
            print("ERROR: Model is not defined. Ensure the model checkpoint was loaded successfully.")
            break

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

if len(all_preds) > 0:
    print("Inference complete.")

## 5. Calculate Evaluation Metrics

In [ ]:
if len(all_preds) > 0:
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')

    print("-" * 30)
    print("EVALUATION METRICS (Single Model)")
    print("-" * 30)
    print(f"Overall Accuracy:  {acc:.4f}")
    print(f"Macro F1-Score:    {macro_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_targets, all_preds, digits=4))
else:
    print("No predictions were generated. Check earlier steps.")

## 6. Visualize Evaluation Results

In [ ]:
if len(all_preds) > 0:
    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Single Model Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()